<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_26_pytest_testing/note_lesson_26_pytest_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 26 — Тестування з pytest (+ валідація AI-коду)

У п'ятницю хтось «підчистив» тариф таксі «Смачно + Таксі» — і мінімальна ціна 80 грн перестала діяти. Руками перевірили одну поїздку на 10 км, і вона була правильною. Сьогодні пишемо перевірки, які повторюються після кожної зміни.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 1 | перший тест, AAA, як pytest знаходить тести, читання падіння, тест без `assert` | 1.1–1.5 | |
| 2 | межі, `pytest.raises` (з `match` і `excinfo`), `DID NOT RAISE`, `pytest.approx` | 2.1–2.5 | **Задача 1** |
| 3 | `parametrize`: таблиця випадків, `ids`, `-k`, декілька декораторів, дві реалізації, `xfail` | 3.1–3.6 | |
| 4 | fixtures: свіжі дані, `conftest.py`, фабрика, інтеграційний тест, `yield` і `tmp_path`, `scope` | 4.1–4.7 | |
| 5 | тести в стилі `unittest` | 5.1–5.2 | |
| 6 | `Mock`, `return_value`, `spec`, `side_effect`, fake, `patch` і `monkeypatch` | 6.1–6.6 | **Задача 2** |
| 7 | coverage: що показує і чого не гарантує | — | |
| 8 | перевірка AI-коду: властивості, тест, що повторює формулу | 8.1–8.4 | **Задача 3** |

**Як працює ноутбук**

- Код і тести записуються у файли папки `lab26/` (клітинка з `%%writefile`), а функція `run_pytest` запускає pytest у цій папці — так само, як ти запускатимеш його в терміналі.
- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (файл з кодом чи тестами, `def`, клас) лише створює його — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Час виконання в рядку `passed in …` у тебе буде іншим — це нормально.
- Теорія й архітектура — у книзі: [Урок 26. Тестування з pytest](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_26/).

## 🔁 Пригадай (без підглядання)

1. Що робить `assert`, якщо умова хибна?
2. Як у командному проєкті уроку 16 тести однієї задачі не залежали від інших?
3. Чому стадії конвеєра з уроку 25 легко перевіряти окремо?

<details>
<summary>Відповіді</summary>

1. Кидає `AssertionError`.
2. Замість функцій інших учасників підставлялися готові заглушки.
3. Кожна стадія приймає будь-яке ітерабельне — досить списку з кількох рядків.

</details>

## ⚙️ Підготовка

Якщо pytest не встановлено, виконай `!pip install pytest` (у Colab він уже є).

`run_pytest(*args)` запускає `python -m pytest -q` у папці `lab26/` окремим процесом, друкує кінець звіту й повертає **код виходу**: `0` — усі тести пройшли, `1` — є падіння, `5` — не знайдено жодного тесту. Тому перевірки нижче мають вигляд `assert run_pytest(...) == 0`.

In [ ]:
import os
import subprocess
import sys

LAB = "lab26"
os.makedirs(LAB, exist_ok=True)
if LAB not in sys.path:
    sys.path.insert(0, LAB)          # щоб імпортувати модулі з lab26/ і в самому ноутбуці


def run_pytest(*args):
    """Запускає pytest у папці LAB і повертає код виходу: 0 — усі тести пройшли."""
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", *args],
        cwd=LAB, capture_output=True, text=True,
    )
    print((result.stdout or result.stderr)[-1500:])
    return result.returncode

---
## 1. Перший тест

### Приклад 1.1. Код, який перевіряємо

Тариф — подача `base`, ціна кілометра `per_km` і мінімальна вартість `min_fare`. `fare` рахує поїздку й не дає їй коштувати менше за мінімум; `apply_promo` застосовує знижку від 0 до 50 %. Записуємо модуль у файл `pricing.py`:

In [ ]:
%%writefile lab26/pricing.py
"""Тарифи й знижки."""

from dataclasses import dataclass


@dataclass(frozen=True)
class Tariff:
    base: int       # подача, грн
    per_km: int     # грн за кілометр
    min_fare: int   # мінімальна вартість поїздки, грн


def fare(km, tariff):
    """Вартість поїздки на km кілометрів, не менша за мінімальну."""
    if km < 0:
        raise ValueError(f"відстань не може бути від'ємною: {km}")
    return max(tariff.min_fare, round(tariff.base + tariff.per_km * km))


def apply_promo(amount, percent):
    """Сума після знижки percent % (від 0 до 50), округлена до гривні."""
    if not 0 <= percent <= 50:
        raise ValueError(f"знижка має бути від 0 до 50 %, а маємо {percent}")
    return round(amount * (100 - percent) / 100)

Так ми перевіряли б «руками» — викликати й подивитися:

In [ ]:
from pricing import Tariff, apply_promo, fare

day = Tariff(base=40, per_km=12, min_fare=80)
print(fare(10, day), fare(0.5, day), apply_promo(340, 10))

`160 80 306`: 40 + 12 × 10 = 160; поїздка на 0,5 км коштувала б 46, але мінімум — 80; 340 зі знижкою 10 % — 306. Ручна перевірка правильна, але її треба повторювати після кожної зміни. Тест робить це за нас.

### Приклад 1.2. Тест — функція `test_…` з `assert`

Тест для pytest — звичайна функція з назвою `test_…` у файлі `test_….py`. Усередині — схема **AAA**: Arrange (підготувати дані) → Act (виконати дію) → Assert (перевірити результат).

In [ ]:
%%writefile lab26/test_first.py
from pricing import Tariff, fare


def test_long_trip_is_base_plus_distance():
    # Arrange
    tariff = Tariff(base=40, per_km=12, min_fare=80)
    # Act
    price = fare(10, tariff)
    # Assert
    assert price == 160


def test_short_trip_costs_min_fare():
    assert fare(0.5, Tariff(base=40, per_km=12, min_fare=80)) == 80

In [ ]:
assert run_pytest("test_first.py") == 0

Дві крапки — два пройдені тести, `2 passed`. Жодного `main()`, жодної реєстрації: pytest сам знайшов функції `test_…`.

### Приклад 1.3. Як pytest знаходить тести

Правила збору: файли `test_*.py` (або `*_test.py`), у них — функції `test_*` і методи `test_*` у класах, чия назва починається з `Test`. Перевіримо: у файлі нижче є функція, що впала б (`check_…`), і клас з падінням (`ShortTripChecks`), — але їх pytest не збере.

In [ ]:
%%writefile lab26/test_discovery.py
from pricing import Tariff, fare

DAY = Tariff(base=40, per_km=12, min_fare=80)


def test_ten_km():
    assert fare(10, DAY) == 160


def check_ten_km():                 # не test_… — pytest її не запускає
    assert fare(10, DAY) == 999


class TestShortTrips:               # клас Test… — методи test_… зібрано
    def test_half_km(self):
        assert fare(0.5, DAY) == 80

    def test_zero_km(self):
        assert fare(0, DAY) == 80


class ShortTripChecks:              # назва не з Test — клас пропущено
    def test_never_runs(self):
        assert False

In [ ]:
assert run_pytest("-v", "test_discovery.py") == 0

`-v` друкує кожен тест окремо: `test_ten_km`, `TestShortTrips::test_half_km`, `TestShortTrips::test_zero_km` — **3 passed**. `check_ten_km` і `ShortTripChecks` не потрапили у звіт. **Правило:** тест, який названо не за правилами, не «падає» — він просто ніколи не запускається, тож помилку в назві легко не помітити.

### Приклад 1.4. Читаємо падіння

Нижче — «п'ятнична» версія тарифу без `max`. Обидва тести написано так, як вимагає бізнес.

**🔮 Прогноз:** який із двох тестів упаде і що покаже pytest у рядку `assert`?

In [ ]:
%%writefile lab26/test_broken_demo.py
from pricing import Tariff


def fare_after_friday(km, tariff):
    return round(tariff.base + tariff.per_km * km)


def test_long_trip():
    assert fare_after_friday(10, Tariff(40, 12, 80)) == 160


def test_short_trip():
    assert fare_after_friday(0.5, Tariff(40, 12, 80)) == 80

In [ ]:
assert run_pytest("test_broken_demo.py") != 0, "тут падіння — це очікувано"

<details>
<summary>Відповідь</summary>

Падає лише `test_short_trip`: `assert 46 == 80`, а рядок `where` показує, звідки взялося 46. Поїздка на 10 км проходить — саме її й перевірили руками в п'ятницю.

</details>

Як читати звіт: `.` — пройдений тест, `F` — провалений; рядок з `>` — де саме впало; `E assert 46 == 80` — pytest **розібрав** звичайний `assert` і показав обидва значення; останній блок `short test summary info` — короткий список: який тест і чому.

### Приклад 1.5. Тест без `assert` проходить завжди

Типова помилка: порівняння написали, а `assert` — забули. Перевіримо той самий зламаний тариф:

In [ ]:
%%writefile lab26/test_no_assert.py
from pricing import Tariff


def fare_after_friday(km, tariff):
    return round(tariff.base + tariff.per_km * km)


def test_short_trip_without_assert():
    fare_after_friday(0.5, Tariff(40, 12, 80)) == 80      # результат порівняння просто викидається

In [ ]:
assert run_pytest("test_no_assert.py") == 0

`1 passed` — на зламаному коді! Вираз `... == 80` обчислився в `False` і загубився. **Правило:** тест мусить щось стверджувати (`assert`, `pytest.raises`, `assert_called…`). А щоб переконатися, що тест справжній, його хоча б раз запускають на зламаному коді й бачать падіння (так ми робитимемо в задачах).

---
## 2. Межі, винятки, дробові числа

### Приклад 2.1. Де межа мінімуму

Помилки живуть на **межах**: нуль, рівно мінімум, трохи нижче й трохи вище. Для денного тарифу межа там, де `40 + 12 × km` дорівнює 80, тобто при 3⅓ км. Функція нижче для кожної відстані друкує «сиру» ціну без мінімуму і ціну `fare`:

In [ ]:
def boundary_rows(tariff, distances):
    """Рядки (km, ціна без мінімуму, fare) — щоб побачити, де вмикається мінімум."""
    return [(km, tariff.base + tariff.per_km * km, fare(km, tariff)) for km in distances]

In [ ]:
for km, raw, price in boundary_rows(day, [0, 3, 3.3, 3.4, 3.5, 10]):
    note = "мінімум" if price == day.min_fare and raw < day.min_fare else ""
    print(f"{km:>4} км   без мінімуму {raw:6.1f}   fare = {price:3}   {note}")

До 3,3 км включно (79,6 → округлено 80) працює мінімум, з 3,4 км (80,8 → 81) — уже формула. Отже, тести тарифу мають бути **по обидва боки** межі: 0, 3, 3,5 км — і одна «довга» поїздка. Тест лише на 10 км п'ятничний баг не побачив би.

### Приклад 2.2. `pytest.raises`: виняток — теж результат

Від'ємна відстань — неможливе значення, і `fare` має кинути `ValueError`. `pytest.raises` — контекстний менеджер: тест проходить, лише якщо всередині `with` виникне саме цей виняток. `match` — регулярний вираз, який шукається в тексті помилки; `as excinfo` дає доступ до самого винятку.

In [ ]:
%%writefile lab26/test_raises.py
import pytest

from pricing import Tariff, fare

DAY = Tariff(base=40, per_km=12, min_fare=80)


def test_negative_distance_is_rejected():
    with pytest.raises(ValueError, match="від'ємною"):
        fare(-1, DAY)


def test_error_message_contains_value():
    with pytest.raises(ValueError) as excinfo:
        fare(-2.5, DAY)
    assert "-2.5" in str(excinfo.value)

In [ ]:
assert run_pytest("-v", "test_raises.py") == 0

Обидва тести пройшли: `fare(-1, …)` кинув `ValueError` зі словом «від'ємною», а в `excinfo.value` лежить сам виняток — можна перевірити й текст повідомлення.

### Приклад 2.3. `DID NOT RAISE`

Що буде, якщо перевірку `km < 0` хтось прибере? Функція `fare_without_check` нижче — саме така версія.

In [ ]:
%%writefile lab26/test_did_not_raise.py
import pytest

from pricing import Tariff

DAY = Tariff(base=40, per_km=12, min_fare=80)


def fare_without_check(km, tariff):
    return max(tariff.min_fare, round(tariff.base + tariff.per_km * km))


def test_negative_distance_is_rejected():
    with pytest.raises(ValueError):
        fare_without_check(-1, DAY)

In [ ]:
assert run_pytest("--tb=short", "test_did_not_raise.py") == 1

`Failed: DID NOT RAISE <class 'ValueError'>` — виняток очікували, а функція спокійно повернула 80. `--tb=short` скорочує звіт до головного.

### Приклад 2.4. У `with` — лише один рядок

А тепер той самий зламаний `fare_without_check`, але в `with` потрапили **два** рядки. Перший рядок кидає `ValueError` з іншої причини — і до перевірки `fare` справа не доходить.

In [ ]:
%%writefile lab26/test_raises_scope.py
import pytest

from pricing import Tariff, apply_promo

DAY = Tariff(base=40, per_km=12, min_fare=80)


def fare_without_check(km, tariff):
    return max(tariff.min_fare, round(tariff.base + tariff.per_km * km))


def test_two_lines_hide_the_bug():
    with pytest.raises(ValueError):
        price = apply_promo(340, 60)          # кидає ValueError: знижка понад 50 %
        fare_without_check(-1, DAY)           # цей рядок не виконується ніколи

In [ ]:
assert run_pytest("test_raises_scope.py") == 0

`1 passed` на зламаному коді: виняток кинув не той рядок. **Правило:** усередині `pytest.raises` — лише той один виклик, що **повинен** кинути виняток; підготовку пиши до `with`.

### Приклад 2.5. Дробові числа: `pytest.approx`

Дробові числа в комп'ютері зберігаються наближено, тому `==` для них підступний:

In [ ]:
checks = [340, 305, 410]
print(0.1 + 0.2 == 0.3, 0.1 + 0.2)
print(sum(checks) / len(checks))

`0.1 + 0.2` — це `0.30000000000000004`, а середній чек — `351.666…`. `pytest.approx` порівнює з допуском: `abs` — абсолютним, `rel` — відносним (частка від значення); працює і для списків.

In [ ]:
%%writefile lab26/test_approx.py
import pytest


def test_float_sum():
    assert 0.1 + 0.2 == pytest.approx(0.3)


def test_average_check_abs():
    checks = [340, 305, 410]
    assert sum(checks) / len(checks) == pytest.approx(351.67, abs=0.01)


def test_average_check_rel():
    assert 351.6666 == pytest.approx(350, rel=0.01)          # 1 % від 350 — це 3,5


def test_list_of_floats():
    assert [0.1 + 0.2, 0.2 + 0.4] == pytest.approx([0.3, 0.6])

In [ ]:
assert run_pytest("test_approx.py") == 0

Усі 4 пройшли. Без `approx` перший і останній упали б через `0.30000000000000004` і `0.6000000000000001`.

### Задача 1. Знайди баг тестом

Правило: доставка безкоштовна для замовлень **від 500 грн** (включно), інакше — 60 грн. Напиши `test_fee.py` для `delivery_fee(total)` з модуля `fee`: випадки з **обох боків межі** (і саму межу), плюс нуль і велике замовлення. Зручно — один `parametrize` (розділ 3) або кілька окремих тестів.

Перевірка в наступній клітинці запустить твої тести **двічі**: на версії з помилкою (`>` замість `>=` — вони мусять упасти) і на правильній (мусять пройти).

In [ ]:
%%writefile lab26/test_fee.py
import pytest

from fee import delivery_fee

# YOUR CODE HERE


In [ ]:
BUGGY = "def delivery_fee(total):\n    return 0 if total > 500 else 60\n"
CORRECT = "def delivery_fee(total):\n    return 0 if total >= 500 else 60\n"

with open(f"{LAB}/fee.py", "w", encoding="utf-8") as file:
    file.write(BUGGY)
assert run_pytest("test_fee.py") == 1, "твої тести не помітили помилку на межі 500 (або тестів ще немає)"

with open(f"{LAB}/fee.py", "w", encoding="utf-8") as file:
    file.write(CORRECT)
assert run_pytest("test_fee.py") == 0, "на правильному коді тести мають пройти"
print("✅ Задача 1 розв'язана")

---
## 3. `parametrize`: один тест — таблиця випадків

### Приклад 3.1. Таблиця відстаней з `ids`

П'ять майже однакових тестів для п'яти відстаней — копіпаста. `@pytest.mark.parametrize("km, expected", [...])` описує випадки таблицею, і pytest запускає тест для кожного рядка окремо. `ids` дає рядкам зрозумілі імена.

In [ ]:
%%writefile lab26/test_fare_table.py
import pytest

from pricing import Tariff, fare

DAY = Tariff(base=40, per_km=12, min_fare=80)


@pytest.mark.parametrize(
    "km, expected",
    [
        (0, 80),       # нуль кілометрів — усе одно мінімум
        (3, 80),       # 40 + 36 = 76 < 80
        (3.5, 82),     # 40 + 42 = 82 — вже більше за мінімум
        (10, 160),
        (2.25, 80),    # 40 + 27 = 67
    ],
    ids=["zero", "below-min", "just-above-min", "long", "fraction"],
)
def test_fare_table(km, expected):
    assert fare(km, DAY) == expected

In [ ]:
assert run_pytest("-v", "test_fare_table.py") == 0

П'ять окремих тестів: `test_fare_table[zero]`, `[below-min]`, … Падіння одного рядка не зупиняє решту, а в звіті одразу видно, який випадок зламався. Новий випадок — новий рядок таблиці.

### Приклад 3.2. `-k`: запустити частину тестів

`-k` залишає лише тести, в імені яких є фрагмент (разом з `id` у квадратних дужках). Файл той самий — змінюємо лише запуск:

In [ ]:
assert run_pytest("-v", "test_fare_table.py", "-k", "min") == 0

`2 passed, 3 deselected`: «min» є лише в `below-min` і `just-above-min`. Зручно, коли треба швидко перезапустити один зламаний випадок.

### Приклад 3.3. Тести для промокоду: `parametrize` + `raises`

`apply_promo(amount, percent)` приймає знижку від 0 до 50 % і округлює до гривні. Звичайні випадки — таблицею (серед них 0 % — межа «без знижки»), межа 50 % — окремим тестом, недопустимі значення −5 і 51 — параметризованим `pytest.raises`. Зверни увагу: у таблиці недопустимих значень лише один стовпчик, `"percent"`.

In [ ]:
%%writefile lab26/test_promo.py
import pytest

from pricing import apply_promo


@pytest.mark.parametrize("amount, percent, expected", [(340, 10, 306), (340, 0, 340), (95, 15, 81)])
def test_apply_promo(amount, percent, expected):
    assert apply_promo(amount, percent) == expected


def test_max_promo_is_half():
    assert apply_promo(200, 50) == 100


@pytest.mark.parametrize("percent", [-5, 51])
def test_bad_percent(percent):
    with pytest.raises(ValueError):
        apply_promo(340, percent)

In [ ]:
assert run_pytest("-v", "test_promo.py") == 0

6 тестів: 3 рядки таблиці, межа 50 %, два недопустимі відсотки. 95 зі знижкою 15 % — це 80,75, тобто 81 після округлення.

### Приклад 3.4. Два декоратори — усі комбінації

Якщо поставити два `parametrize` один над одним, pytest перебере **усі пари**: 3 відстані × 2 тарифи = 6 тестів. Перевіряємо **властивість**, що не залежить від конкретних чисел: поїздка ніколи не дешевша за мінімум свого тарифу.

In [ ]:
%%writefile lab26/test_matrix.py
import pytest

from pricing import Tariff, fare

DAY = Tariff(base=40, per_km=12, min_fare=80)
NIGHT = Tariff(base=60, per_km=15, min_fare=120)


@pytest.mark.parametrize("km", [0, 3, 10])
@pytest.mark.parametrize("tariff", [DAY, NIGHT], ids=["day", "night"])
def test_never_below_min_fare(km, tariff):
    assert fare(km, tariff) >= tariff.min_fare

In [ ]:
assert run_pytest("-v", "test_matrix.py") == 0

Імена на кшталт `test_never_below_min_fare[day-0]`, `[night-10]` — по одному на кожну пару. Таблиця 3 × 2 описана двома рядками коду.

### Приклад 3.5. Одна таблиця — дві реалізації

`parametrize` може перебирати навіть **функції**. Проженемо ту саму таблицю відстаней через правильний `fare` і через «п'ятничну» версію — і побачимо, на яких саме випадках вони розходяться. `--tb=no` прибирає подробиці падінь, лишаючи підсумок.

In [ ]:
%%writefile lab26/test_two_versions.py
import pytest

from pricing import Tariff, fare

DAY = Tariff(base=40, per_km=12, min_fare=80)


def fare_after_friday(km, tariff):
    return round(tariff.base + tariff.per_km * km)


@pytest.mark.parametrize("impl", [fare, fare_after_friday], ids=["correct", "friday"])
@pytest.mark.parametrize("km, expected", [(0, 80), (3, 80), (3.5, 82), (10, 160)],
                         ids=["zero", "below-min", "above-min", "long"])
def test_fare(impl, km, expected):
    assert impl(km, DAY) == expected

In [ ]:
assert run_pytest("--tb=no", "test_two_versions.py") == 1

`2 failed, 6 passed`: «п'ятнична» версія падає лише на `zero` і `below-min` — нижче межі мінімуму — і проходить на `above-min` та `long`. Саме тому розробник у п'ятницю нічого не помітив.

### Приклад 3.6. Відомий баг: `pytest.param` і `xfail`

Буває, що баг знайдено, а виправити зараз не можна. Тоді випадок позначають `xfail` («очікуване падіння»): він не ламає збірку, але залишається у звіті як нагадування. Окремий рядок таблиці з власною позначкою пишуть через `pytest.param`.

In [ ]:
%%writefile lab26/test_xfail.py
import pytest

from pricing import Tariff

DAY = Tariff(base=40, per_km=12, min_fare=80)


def fare_after_friday(km, tariff):
    return round(tariff.base + tariff.per_km * km)


@pytest.mark.parametrize("km, expected", [
    (10, 160),
    pytest.param(0.5, 80, id="short", marks=pytest.mark.xfail(reason="загублено мінімум, задача DEV-17")),
])
def test_friday_fare(km, expected):
    assert fare_after_friday(km, DAY) == expected

In [ ]:
assert run_pytest("-v", "-rx", "test_xfail.py") == 0

`1 passed, 1 xfailed`: короткий випадок позначено `XFAIL` з причиною (її показує `-rx`), код виходу — 0. Коли баг виправлять, тест почне проходити — і pytest повідомить `XPASS`: час прибрати позначку.

---
## 4. Fixtures: дані для тестів

### Приклад 4.1. Fixture за іменем параметра — і щоразу свіжа

**Fixture** — функція з `@pytest.fixture`. pytest викликає її **перед кожним тестом**, що має параметр з таким самим ім'ям, і передає результат. Перевіримо дві речі: як fixture потрапляє в тест і що кожен тест отримує **новий** об'єкт. `-s` вимикає перехоплення `print`, щоб побачити, коли fixture викликається.

In [ ]:
%%writefile lab26/test_fixture_basics.py
import pytest

from pricing import Tariff, fare


@pytest.fixture
def day_tariff():
    print("\n  [fixture] створюю day_tariff")
    return Tariff(base=40, per_km=12, min_fare=80)


@pytest.fixture
def basket():
    return []


def test_long_trip(day_tariff):
    assert fare(10, day_tariff) == 160


def test_short_trip(day_tariff):
    assert fare(0.5, day_tariff) == 80


def test_basket_first(basket):
    basket.append("борщ")
    assert basket == ["борщ"]


def test_basket_second(basket):
    assert basket == []          # новий список: перший тест на нього не вплинув

In [ ]:
assert run_pytest("-s", "test_fixture_basics.py") == 0

Рядок `[fixture] створюю day_tariff` надруковано **двічі** — по разу для кожного тесту, якому потрібен `day_tariff`. А `test_basket_second` бачить порожній кошик, хоча `test_basket_first` щось у нього поклав: тести не залежать одне від одного.

### Приклад 4.2. `conftest.py` і fixture, що приймає інші fixtures

Fixtures, потрібні в кількох файлах, кладуть у `conftest.py`: pytest знаходить його сам, імпортувати не треба. Fixture може приймати інші fixtures так само, як тест. Додамо нічний тариф — подача 60 грн, 15 грн/км, мінімум 120 грн. Межа мінімуму: `60 + 15 × km = 120` при 4 км.

In [ ]:
%%writefile lab26/conftest.py
"""Спільні fixtures для всіх тестів у lab26/."""
import pytest

from pricing import Tariff


@pytest.fixture
def day_tariff():
    return Tariff(base=40, per_km=12, min_fare=80)


@pytest.fixture
def night_tariff():
    return Tariff(base=60, per_km=15, min_fare=120)


@pytest.fixture
def tariffs(day_tariff, night_tariff):           # fixture з двох інших fixtures
    return {"day": day_tariff, "night": night_tariff}

In [ ]:
%%writefile lab26/test_night.py
import pytest

from pricing import fare


@pytest.mark.parametrize("km, expected", [(0, 120), (4, 120), (4.5, 128), (10, 210)],
                         ids=["zero", "on-boundary", "above", "long"])
def test_night_fare_table(km, expected, night_tariff):
    assert fare(km, night_tariff) == expected


@pytest.mark.parametrize("km", [0, 2, 5, 12])
def test_night_is_never_cheaper(km, tariffs):
    assert fare(km, tariffs["night"]) >= fare(km, tariffs["day"])

In [ ]:
assert run_pytest("-v", "test_night.py") == 0

8 тестів пройшли, хоча в `test_night.py` немає жодного імпорту fixtures: pytest узяв `night_tariff` і `tariffs` з `conftest.py`. Параметри таблиці й fixtures легко поєднуються в одному тесті. (Файл `test_fixture_basics.py` має власний `day_tariff` — локальна fixture перекриває однойменну з `conftest.py`.)

### Приклад 4.3. Конвеєр подій: код під тест

Конвеєр кур'єрських подій з уроку 25 — у файл `events.py`. Кожна стадія приймає ітерабельне й віддає ітерабельне: `parse` → `in_shift` → `durations` → `report`.

In [ ]:
%%writefile lab26/events.py
"""Конвеєр подій кур'єрів з уроку 25: кожна стадія — «ітерабельне → ітерабельне»."""

from itertools import dropwhile


def minutes(hhmm):
    hours, mins = hhmm.split(":")
    return int(hours) * 60 + int(mins)


def parse(lines, rejected):
    for line in lines:
        parts = line.split()
        if len(parts) != 4 or not parts[2].isdigit():
            rejected.append(line)
            continue
        time, courier, order, kind = parts
        yield {"time": minutes(time), "courier": courier, "order": int(order), "kind": kind}


def in_shift(events, start):
    return dropwhile(lambda event: event["time"] < start, events)


def durations(events):
    picked = {}
    for event in events:
        if event["kind"] == "picked":
            picked[event["order"]] = event["time"]
        elif event["kind"] == "delivered" and event["order"] in picked:
            yield event["courier"], event["time"] - picked.pop(event["order"])


def report(pairs):
    by_courier = {}
    for courier, spent in pairs:
        by_courier.setdefault(courier, []).append(spent)
    return {courier: round(sum(spent) / len(spent), 1) for courier, spent in sorted(by_courier.items())}


def read_log(path):
    with open(path, encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line:
                yield line

Спершу подивимося, що робить конвеєр, на маленькому журналі прямо в ноутбуці:

In [ ]:
from events import durations, in_shift, minutes, parse, report

shift_log = [
    "17:58 D-1 97 delivered",
    "18:03 D-1 101 picked",
    "18:05 D-2 102 picked",
    "18:21 D-1 101 delivered",
    "18:24 ?? зламаний рядок",
    "18:29 D-2 102 delivered",
]
rejected = []
pairs = list(durations(in_shift(parse(shift_log, rejected), minutes("18:00"))))
print(pairs)
print(report(pairs))
print(rejected)

Замовлення 101 доставлено за 18 хв, 102 — за 24; подію 17:58 відкинув `in_shift` (до початку зміни), а зламаний рядок `parse` відклав у `rejected`. Ці числа й будуть очікуваними значеннями тестів.

### Приклад 4.4. Fixture-фабрика: лише важливі поля

Тестам конвеєра потрібні рядки подій, але кожному тесту важливі різні поля. Fixture-**фабрика** повертає не дані, а **функцію**, що створює подію з розумними значеннями за замовчуванням. Тест вказує лише те, що для нього важливо.

In [ ]:
%%writefile lab26/test_events_lab.py
import pytest

from events import durations, parse


@pytest.fixture
def make_event():
    def factory(order=1, kind="picked", time="18:00", courier="D-1"):
        return f"{time} {courier} {order} {kind}"
    return factory


def test_duration_of_one_order(make_event):
    lines = [make_event(kind="picked", time="18:00"), make_event(kind="delivered", time="18:25")]
    assert list(durations(parse(lines, []))) == [("D-1", 25)]


def test_delivered_without_picked_is_skipped(make_event):
    assert list(durations(parse([make_event(kind="delivered")], []))) == []


def test_broken_line_is_rejected(make_event):
    rejected = []
    events = list(parse([make_event(), "битий рядок"], rejected))
    assert len(events) == 1 and rejected == ["битий рядок"]

In [ ]:
assert run_pytest("-v", "test_events_lab.py") == 0

Три тести, кожен — про одну поведінку: тривалість одного замовлення (25 хв), `delivered` без `picked` пропускається, битий рядок іде в `rejected`. Якщо формат подій зміниться, правити доведеться лише фабрику.

### Приклад 4.5. Ледачий `parse` і весь конвеєр разом

`test_parse_is_lazy` фіксує поведінку, яку легко зламати непомітно: генератор нічого не читає, доки його не попросять. `test_whole_pipeline` — **інтеграційний** тест: усі стадії разом на журналі з fixture `log_lines`. `test_read_log_from_file` пише журнал у тимчасову папку `tmp_path` (вбудована fixture pytest — окрема папка для кожного тесту).

In [ ]:
%%writefile lab26/test_pipeline.py
import pytest

from events import durations, in_shift, minutes, parse, read_log, report


@pytest.fixture
def log_lines():
    return [
        "17:58 D-1 97 delivered",
        "18:03 D-1 101 picked",
        "18:05 D-2 102 picked",
        "18:21 D-1 101 delivered",
        "18:24 ?? зламаний рядок",
        "18:29 D-2 102 delivered",
    ]


def test_parse_is_lazy():
    rejected = []
    stream = parse(["зовсім не подія"], rejected)
    assert rejected == []          # генератор ще нічого не читав
    assert list(stream) == []
    assert rejected == ["зовсім не подія"]


def test_whole_pipeline(log_lines):
    rejected = []
    result = report(durations(in_shift(parse(log_lines, rejected), minutes("18:00"))))
    assert result == {"D-1": 18.0, "D-2": 24.0}
    assert len(rejected) == 1


def test_read_log_from_file(tmp_path, log_lines):
    path = tmp_path / "shift.log"
    path.write_text("\n".join(log_lines) + "\n\n", encoding="utf-8")
    assert list(read_log(path)) == log_lines       # порожні рядки наприкінці відкинуто

In [ ]:
assert run_pytest("-v", "test_pipeline.py") == 0

Три пройдені тести. Якщо хтось перепише `parse` так, що він збиратиме список одразу, впаде `test_parse_is_lazy`; якщо зламається будь-яка стадія — `test_whole_pipeline`. А файл з `tmp_path` зникне разом з тимчасовою папкою, не засмічуючи проєкт.

### Приклад 4.6. Прибирання: `yield` у fixture

Fixture з `yield` ділиться на дві частини: до `yield` — підготовка, після — прибирання. Прибирання виконується, **навіть якщо тест упав**. Другий тест нижче падає навмисно — подивимося, чи закриється журнал.

In [ ]:
%%writefile lab26/test_yield.py
import pytest


@pytest.fixture
def shift_log(tmp_path):
    path = tmp_path / "shift.log"
    file = open(path, "w", encoding="utf-8")
    print("\n  відкрив журнал")
    yield file
    file.close()
    print("  закрив журнал")


def test_write_event(shift_log):
    shift_log.write("18:03 D-1 101 picked\n")
    print("  пишу подію")
    assert not shift_log.closed


def test_that_fails(shift_log):
    print("  тест падає")
    assert 1 == 2

In [ ]:
assert run_pytest("-s", "--tb=no", "test_yield.py") == 1

Порядок у виводі: «відкрив журнал» → «пишу подію» → `.` → «закрив журнал»; і для другого тесту так само — «закрив журнал» надруковано **після** `F`. Отже, ресурс звільнено навіть при падінні тесту — те саме, що `with` чи `try/finally`.

### Приклад 4.7. `scope="module"`: тести, що залежать від порядку

Fixture можна створювати рідше: `scope="module"` — одна на весь файл. Це прискорює дорогу підготовку (з'єднання з базою), але для **змінюваних** даних небезпечно:

In [ ]:
%%writefile lab26/test_scope_bug.py
import pytest

from events import parse


@pytest.fixture(scope="module")
def rejected():
    return []


def test_parse_one(rejected):
    list(parse(["bad"], rejected))
    assert rejected == ["bad"]


def test_parse_two(rejected):
    list(parse(["bad too"], rejected))
    assert rejected == ["bad too"]

In [ ]:
print("весь файл:")
assert run_pytest("--tb=line", "test_scope_bug.py") == 1
print("лише другий тест:")
assert run_pytest("test_scope_bug.py", "-k", "two") == 0

Разом другий тест падає — він бачить `['bad', 'bad too']`, бо список спільний. Окремо — проходить. Такі тести «то падають, то ні» залежно від того, що запустили. **Правило:** для даних — scope за замовчуванням (`function`), ширший — лише для незмінних дорогих ресурсів.

---
## 5. `unittest`: тести в стилі класів

### Приклад 5.1. `TestCase`, `setUp`, `assert…`-методи

У стандартній бібліотеці є модуль `unittest`: тести — методи класу-нащадка `TestCase`, перевірки — методи `assertEqual`, `assertRaises`, `assertAlmostEqual`. `setUp` виконується перед кожним тестом — як fixture. Цей стиль зустрінеш у старих проєктах і в Django (`django.test.TestCase` — нащадок `unittest.TestCase`).

In [ ]:
%%writefile lab26/test_unittest_style.py
import unittest

from pricing import Tariff, apply_promo, fare


class FareTest(unittest.TestCase):
    def setUp(self):
        self.tariff = Tariff(base=40, per_km=12, min_fare=80)

    def test_long_trip(self):
        self.assertEqual(fare(10, self.tariff), 160)

    def test_short_trip(self):
        self.assertEqual(fare(0.5, self.tariff), 80)

    def test_negative_distance(self):
        with self.assertRaises(ValueError):
            fare(-1, self.tariff)

    def test_average_check(self):
        self.assertAlmostEqual(sum([340, 305, 410]) / 3, 351.67, places=2)

    def test_promo(self):
        self.assertEqual(apply_promo(340, 10), 306)


if __name__ == "__main__":
    unittest.main()

In [ ]:
assert run_pytest("-v", "test_unittest_style.py") == 0

pytest запускає й такі тести — 5 passed. Тож обирати «або-або» не доводиться: старі `unittest`-тести працюють поруч з новими.

### Приклад 5.2. Той самий файл — через `unittest`

Без pytest ці тести запускає сам Python: `python -m unittest -v <модуль>`. Звіт `unittest` друкує в потік помилок (stderr).

In [ ]:
result = subprocess.run([sys.executable, "-m", "unittest", "-v", "test_unittest_style"],
                        cwd=LAB, capture_output=True, text=True)
print(result.stderr)

`Ran 5 tests … OK`. Звичайні pytest-тести (`def test_…` з голим `assert`) так не запустяться — `unittest` шукає лише класи `TestCase`. Для нового коду pytest зручніший: звичайний `assert`, fixtures за іменем, `parametrize`.

---
## 6. Mock і patch: підміна зовнішнього світу

### Приклад 6.1. `Mock` запам'ятовує кожен виклик

Коли кур'єр забирає замовлення, клієнт отримує SMS. Справжній шлюз ходить у мережу й бере гроші. У тестах такі **межі системи** підміняють. Найпростіше — коли залежність приходить **параметром** (`gateway`):

In [ ]:
%%writefile lab26/notify.py
"""Сповіщення клієнта. Шлюз SMS передається параметром — його легко підмінити в тесті."""


class SmsGateway:
    """Справжній шлюз: ходить у мережу й коштує гроші за кожне повідомлення."""

    def send(self, phone, text):
        raise ConnectionError("справжня мережа недоступна в навчальному проєкті")


def notify_client(order_id, phone, minutes, gateway):
    """Надсилає клієнту час доставки. Повертає True, якщо SMS пішло."""
    text = f"Замовлення №{order_id} буде за {minutes} хв"
    try:
        gateway.send(phone, text)
    except ConnectionError:
        return False
    return True

`unittest.mock.Mock` — об'єкт, у якого є **будь-який** метод; він приймає будь-які аргументи й записує кожен виклик. `side_effect` — виняток, що спрацює під час виклику.

In [ ]:
from unittest.mock import Mock

from notify import SmsGateway, notify_client

gateway = Mock()
print(notify_client(101, "+380501234567", 25, gateway))
print(gateway.send.call_count, gateway.send.call_args)

gateway.send.side_effect = ConnectionError("шлюз не відповідає")
print(notify_client(102, "+380501234567", 30, gateway))
print(notify_client(103, "+380501234567", 30, SmsGateway()))

`True` — Mock «надіслав» SMS; `1 call('+380501234567', 'Замовлення №101 буде за 25 хв')` — кількість викликів і аргументи останнього. Після `side_effect` — `False`: функція впоралася з падінням шлюзу. Справжній `SmsGateway` поводиться так само (`False`), але в реальному житті він спершу витратив би гроші.

### Приклад 6.2. `return_value` і пастка «будь-який метод»

`return_value` задає, що поверне виклик. Але «будь-який метод» має й зворотний бік: Mock не помітить друкарську помилку в назві методу. `Mock(spec=Клас)` дозволяє лише ті атрибути, що є у справжнього класу.

In [ ]:
loose = Mock()
loose.send.return_value = "msg-17"
print(loose.send("+380501234567", "привіт"))
print(loose.sendd("+380501234567", "привіт"))        # друкарська помилка — і жодної реакції

strict = Mock(spec=SmsGateway)
try:
    strict.sendd("+380501234567", "привіт")
except AttributeError as error:
    print("AttributeError:", error)

`loose.send(...)` повернув `msg-17`; `loose.sendd(...)` теж «спрацював» і повернув ще один Mock — тест з такою помилкою пройшов би, хоча справжній код упав би. `Mock(spec=SmsGateway)` одразу кинув `AttributeError`: у шлюзі немає методу `sendd`. **Правило:** якщо є справжній клас, давай Mock-у `spec`.

### Приклад 6.3. `side_effect`-список: перший виклик падає, другий — ні

Якщо `side_effect` — список, кожен наступний виклик бере наступний елемент: виняток кидається, звичайне значення повертається. Так моделюють «мережа то є, то немає».

In [ ]:
flaky = Mock()
flaky.send.side_effect = [ConnectionError("таймаут"), None, ConnectionError("таймаут")]

for order_id in (201, 202, 203):
    print(order_id, notify_client(order_id, "+380671112233", 15, flaky))

print(flaky.send.call_count)
print(flaky.send.call_args_list[1])

`201 False`, `202 True`, `203 False` — точно за списком. `call_count` — 3, а `call_args_list` зберігає **усі** виклики, не лише останній: другий був для замовлення №202.

### Приклад 6.4. Fake: маленький справжній клас замість Mock

Замість Mock можна написати **fake** — клас, що поводиться як шлюз, але складає повідомлення в список. Fake читається простіше, коли важливий стан («що вже надіслано»), і не дозволить викликати неіснуючий метод.

In [ ]:
class FakeGateway:
    """Шлюз для тестів: нічого не надсилає, лише запам'ятовує."""

    def __init__(self, fail=False):
        self.sent = []
        self.fail = fail

    def send(self, phone, text):
        if self.fail:
            raise ConnectionError("fake: мережа вимкнена")
        self.sent.append((phone, text))

In [ ]:
fake = FakeGateway()
notify_client(301, "+380931234567", 20, fake)
notify_client(302, "+380931234567", 35, fake)
print(fake.sent)
print(notify_client(303, "+380931234567", 10, FakeGateway(fail=True)))

У `fake.sent` — два повідомлення в порядку надсилання; шлюз з `fail=True` дає `False`. Перевірка в тесті стає звичайним `assert fake.sent == [...]`.

### Задача 2. Mock SMS-шлюзу в pytest

Напиши `test_notify_lab.py` для `notify_client(order_id, phone, minutes, gateway)` з `unittest.mock.Mock`:

- тест, що SMS надіслано: функція повертає `True`, а текст і номер перевір через `gateway.send.assert_called_once_with(...)`;
- тест «шлюз упав»: через `side_effect = ConnectionError(...)` перевір, що функція повертає `False` і не падає.

Перевірка — у наступній клітинці.

In [ ]:
%%writefile lab26/test_notify_lab.py
from unittest.mock import Mock

from notify import notify_client

# YOUR CODE HERE


In [ ]:
assert run_pytest("test_notify_lab.py") == 0, "тести мають пройти (або тестів ще немає)"
text = open(f"{LAB}/test_notify_lab.py", encoding="utf-8").read()
assert "side_effect" in text and "assert_called_once_with" in text, "потрібні side_effect і assert_called_once_with"
print("✅ Задача 2 розв'язана")

### Приклад 6.5. `patch`: підміняти там, де ім'я **використовують**

Не весь код приймає залежності параметром. Модуль `services.py` імпортує функцію `send_sms` напряму. `unittest.mock.patch("модуль.ім'я")` тимчасово замінює ім'я в модулі на Mock. Головне правило — **patch where used**.

In [ ]:
%%writefile lab26/sms.py
def send_sms(phone, text):
    raise ConnectionError("справжня мережа недоступна в навчальному проєкті")

In [ ]:
%%writefile lab26/services.py
from sms import send_sms


def confirm_order(order_id, phone):
    send_sms(phone, f"Замовлення №{order_id} прийнято")
    return "confirmed"

Два тести: перший підміняє `services.send_sms` (де ім'я використовують), другий — `sms.send_sms` (де його визначили).

🔮 **Прогноз:** який із тестів пройде?

In [ ]:
%%writefile lab26/test_patch.py
from unittest.mock import patch

from services import confirm_order


def test_patch_where_used():
    with patch("services.send_sms") as fake_send:
        assert confirm_order(101, "+380501234567") == "confirmed"
    fake_send.assert_called_once_with("+380501234567", "Замовлення №101 прийнято")


def test_patch_where_defined():
    with patch("sms.send_sms") as fake_send:
        assert confirm_order(101, "+380501234567") == "confirmed"
    fake_send.assert_called_once()

In [ ]:
assert run_pytest("--tb=line", "test_patch.py") == 1

<details>
<summary>Відповідь</summary>

Проходить лише `test_patch_where_used`. Рядок `from sms import send_sms` уже **скопіював посилання** на функцію в простір імен `services` (урок 13). Підміна в `sms` цього посилання не змінює — і `confirm_order` викликає справжню функцію: `ConnectionError: справжня мережа недоступна…`.

</details>

### Приклад 6.6. `monkeypatch` — вбудована fixture pytest

pytest має власну fixture для підмін — `monkeypatch`. `monkeypatch.setattr(об'єкт, "ім'я", нове)` замінює атрибут і **сам повертає** старе значення після тесту. Замість Mock тут — звичайна функція, що пише виклики в список.

In [ ]:
%%writefile lab26/test_monkeypatch.py
import services


def test_confirm_order_with_monkeypatch(monkeypatch):
    sent = []
    monkeypatch.setattr(services, "send_sms", lambda phone, text: sent.append((phone, text)))
    assert services.confirm_order(102, "+380501234567") == "confirmed"
    assert sent == [("+380501234567", "Замовлення №102 прийнято")]


def test_original_is_back():
    import sms
    assert services.send_sms is sms.send_sms        # після попереднього тесту — знову справжня

In [ ]:
assert run_pytest("-v", "test_monkeypatch.py") == 0

Обидва пройшли: у першому тесті `send_sms` замінено лямбдою, а другий бачить справжню функцію — `monkeypatch` прибрав за собою. Правило «patch where used» діє і тут: підміняємо атрибут модуля `services`.

> ⚠️ **Коли mock шкодить.** Мокати варто **межі системи**: мережу, платежі, SMS, пошту, час, випадковість. Якщо мокати власні функції, тест перевіряє не поведінку, а те, **як** код написано: будь-який рефакторинг ламає тести, хоча програма працює.

---
## 7. Coverage: що тести не зачепили

`pytest-cov` (окрема бібліотека: `pip install pytest-cov`) показує, які рядки коду виконувалися під час тестів: `python -m pytest --cov=delivery --cov-report=term-missing`. Непокриті рядки — підказка, де тестів немає, і кожне таке місце треба пояснити (у проєкті `delivery_tests/` це саме справжні виклики мережі, які ми свідомо підмінили).

А от 100 % нічого не гарантує. Рядок вважається покритим, якщо він **виконався**, навіть коли жоден `assert` не перевірив результат. Тест без `assert` з прикладу 1.5 дав би покриття рядка `return` — і пропустив би п'ятничний баг. Якість тестів — у перевірках, а не у відсотку. Повний приклад звіту — у книзі.

---
## 8. Валідація AI-коду

### Приклад 8.1. «Розділи рахунок порівну»

Попросили AI написати функцію, що ділить рахунок компанії порівну між людьми. Ось що він дав:

In [ ]:
def split_bill_ai(total, people):
    """Згенеровано AI: «розділи рахунок порівну між людьми»."""
    share = round(total / people)
    return [share] * people

In [ ]:
print(split_bill_ai(300, 3))
print(split_bill_ai(100, 3), sum(split_bill_ai(100, 3)))
print(split_bill_ai(200, 3), sum(split_bill_ai(200, 3)))

На «зручному» `300 / 3` усе гаразд. А `100 / 3` губить гривню (сума 99), `200 / 3` — додає зайву (67 × 3 = 201). Помилки ховаються не у формулі, а в **округленні**.

### Приклад 8.2. Перевірка на властивості

Щоб не шукати такі випадки навмання, записуємо **специфікацію** — властивості, що мусять виконуватися для **будь-яких** входів:

1. частин стільки, скільки людей;
2. сума частин дорівнює рахунку;
3. частини відрізняються щонайбільше на 1 грн;
4. нуль людей — `ValueError`, а не `ZeroDivisionError`.

`check_split(split)` перевіряє ці правила на сотнях комбінацій і повертає список порушень.

In [ ]:
def check_split(split):
    problems = []
    for total in range(0, 301, 7):
        for people in range(1, 8):
            shares = split(total, people)
            if len(shares) != people:
                problems.append(f"{total} на {people}: частин {len(shares)}")
            elif sum(shares) != total:
                problems.append(f"{total} на {people}: сума {sum(shares)}")
            elif max(shares) - min(shares) > 1:
                problems.append(f"{total} на {people}: різниця {max(shares) - min(shares)}")
    try:
        split(100, 0)
        problems.append("0 людей: помилки немає")
    except ValueError:
        pass
    except ZeroDivisionError:
        problems.append("0 людей: ZeroDivisionError замість ValueError")
    return problems

In [ ]:
problems = check_split(split_bill_ai)
print(len(problems))
print(problems[:3])
print(problems[-1])

151 порушення на 301 комбінації: уже `7 на 2` дає суму 8. І нуль людей падає `ZeroDivisionError` замість зрозумілого `ValueError`.

### Приклад 8.3. Правильна версія — остача по гривні

`divmod(total, people)` дає повну частку й остачу; першим `rest` людям додаємо по гривні.

In [ ]:
def split_bill(total, people):
    if people < 1:
        raise ValueError(f"людей має бути щонайменше 1, а маємо {people}")
    share, rest = divmod(total, people)
    return [share + 1] * rest + [share] * (people - rest)

In [ ]:
print(split_bill(100, 3), split_bill(200, 3), split_bill(7, 2))
print(check_split(split_bill))

`[34, 33, 33] [67, 67, 66] [4, 3]` і порожній список порушень. Такі перевірки називають **тестами властивостей** (property-based): замість «на вході 100 і 3 — на виході [34, 33, 33]» вони перевіряють правила, що не залежать від конкретних чисел.

### Приклад 8.4. Тест від AI, що повторює формулу

Попроси AI «напиши тести для `fare`» — і часто отримаєш тест, що **обчислює очікуване тією самою формулою**, що й код. Перевіримо такий тест на «п'ятничній» версії поруч з тестом, де очікуване взято з **вимог**.

In [ ]:
%%writefile lab26/test_formula_copy.py
from pricing import Tariff

DAY = Tariff(base=40, per_km=12, min_fare=80)


def fare_after_friday(km, tariff):
    return round(tariff.base + tariff.per_km * km)


def test_copies_the_formula():
    km = 3
    assert fare_after_friday(km, DAY) == round(DAY.base + DAY.per_km * km)


def test_expected_from_requirements():
    assert fare_after_friday(3, DAY) == 80        # вимога: 3 км — це мінімум, 80 грн

In [ ]:
assert run_pytest("--tb=line", "test_formula_copy.py") == 1

`test_copies_the_formula` проходить на зламаному коді: помилка у формулі потрапила й в очікуване значення. Падає лише тест з числом із вимог — `assert 76 == 80`. **Правило:** очікувані значення беруться з вимог, а не з реалізації.

> 💡 **Чекліст для AI-коду:** 1) спершу специфікація; 2) тести на межі (0, 1, порожнє, від'ємне, рівно на межі); 3) інваріанти для будь-яких входів; 4) переконайся, що тест **падає** на зламаному коді; 5) очікуване — з вимог; 6) код, який не можеш пояснити, не йде в `main`.

### Задача 3. Тести властивостей для `split_bill`

Перенеси властивості з прикладу 8.2 у pytest: напиши в `test_split.py` тести для `split_bill(total, people)` з модуля `split` для багатьох комбінацій `total` і `people` (підказка: два `parametrize` один над одним, приклад 3.4):

1. частин рівно `people`;
2. сума частин дорівнює `total`;
3. частини відрізняються щонайбільше на 1;
4. `people=0` → `ValueError`.

Перевірка в наступній клітинці запустить твої тести на AI-версії (мусять упасти) і на правильній (мусять пройти).

In [ ]:
%%writefile lab26/test_split.py
import pytest

from split import split_bill

# YOUR CODE HERE


In [ ]:
AI_VERSION = """
def split_bill(total, people):
    share = round(total / people)
    return [share] * people
"""
CORRECT = """
def split_bill(total, people):
    if people < 1:
        raise ValueError("людей має бути щонайменше 1")
    share, rest = divmod(total, people)
    return [share + 1] * rest + [share] * (people - rest)
"""

with open(f"{LAB}/split.py", "w", encoding="utf-8") as file:
    file.write(AI_VERSION)
assert run_pytest("--tb=no", "test_split.py") == 1, "твої тести пропустили помилки AI-версії (або тестів ще немає)"

with open(f"{LAB}/split.py", "w", encoding="utf-8") as file:
    file.write(CORRECT)
assert run_pytest("test_split.py") == 0, "на правильній версії тести мають пройти"
print("✅ Задача 3 розв'язана")

## ✅ Самоперевірка

1. Чому п'ятничний баг не зловив тест на 10 км?
2. Як pytest дізнається, яку fixture передати в тест?
3. Навіщо в перевірці задачі 1 тести запускаються на зламаному коді?
4. Що мокати, а що ні?
5. Чому тест, що повторює формулу реалізації, небезпечний?
6. Що не так з `pytest.raises` навколо двох рядків?
7. Чому `patch("sms.send_sms")` не підміняє функцію в `services`?
8. Чим небезпечна fixture зі `scope="module"`, що повертає список?

<details>
<summary>Відповіді</summary>

1. На 10 км мінімум не діє — баг проявляється лише нижче межі.
2. За іменем параметра тестової функції.
3. Тест, який проходить на будь-якому коді, нічого не перевіряє.
4. Межі системи: мережу, SMS, платежі, час. Власну логіку — ні.
5. Помилка у формулі потрапляє і в очікуване значення — тест проходить на неправильному коді.
6. Виняток може кинути не той рядок, і тест пройде на зламаному коді (приклад 2.4).
7. `from sms import send_sms` скопіював посилання в `services`; підміняти треба там, де ім'я використовується.
8. Список спільний для всіх тестів файлу: результат залежить від порядку запуску (приклад 4.7).

</details>

### Шпаргалка

```python
def test_name():                         # файл test_*.py, функція test_*
    assert fare(10, day) == 160          # AAA: Arrange, Act, Assert

with pytest.raises(ValueError, match="від'ємною"):
    fare(-1, day)                        # лише один рядок усередині
assert 351.666 == pytest.approx(351.67, abs=0.01)

@pytest.mark.parametrize("km, expected", [(0, 80), (10, 160)], ids=["zero", "long"])
@pytest.fixture                          # у tests/conftest.py — для всіх файлів
def day_tariff(): return Tariff(40, 12, 80)

gateway = Mock(); gateway.send.side_effect = ConnectionError()
gateway.send.assert_called_once_with(phone, text)
with patch("delivery.services.send_sms"): ...   # там, де ім'я використовується
```

`python -m pytest -v -k fare` · `-s` (показати print) · `--cov=delivery --cov-report=term-missing`

## Далі

- **Проєкт `delivery_tests/`** у папці уроку — 32 тести; спробуй зламати код за `README.md`.
- **Практикум `basics/`** — 5 файлів від першого `assert` до `parametrize`.
- **Урок 27 — П5. Динамічне програмування**: тестами перевіримо, що швидке рішення збігається з повільним.